In [1]:
import re
import torch
from torch import nn
import torch.nn.functional as F

In [2]:
torch.manual_seed(23)
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print("PyTorch:", torch.__version__, "device:", device)

PyTorch: 2.14.1 device: mps


In [3]:
examples = {
    "分类": ("这家餐厅很好吃", "正面"),
    "序列标注": ("张三在北京工作", ["人名", "介词", "地点", "动词"]),
    "生成": ("输入：Good morning. 目标语言：中文", "早上好。"),
}
for task, pair in examples.items():
    print(task, "输入=", pair[0], "目标=", pair[1])

分类 输入= 这家餐厅很好吃 目标= 正面
序列标注 输入= 张三在北京工作 目标= ['人名', '介词', '地点', '动词']
生成 输入= 输入：Good morning. 目标语言：中文 目标= 早上好。


In [4]:
# 获取词汇表 token id
tokens = ["nlp", "models", "learn", "from", "text"]
vocab = {token: index for index, token in enumerate(tokens)}
sentence = ["models", "learn", "from", "text"]
input_ids = [vocab[token] for token in sentence]
print(input_ids)  # 整数序列；ID 的大小不代表语义

[1, 2, 3, 4]


In [5]:
# 从文本中识别词汇
text = "NLP is useful!"
normalized = text.lower().strip()
pieces = re.findall(r"\w+|[^\w\s]", normalized)
print(normalized, pieces)  # 教学用规则，不是通用生产 tokenizer

nlp is useful! ['nlp', 'is', 'useful', '!']


In [7]:
## 3. 词表、未知词和特殊 token
# 有限词表无法收录所有新词。传统词级方案通常用 `[UNK]`（unknown）表示未登录词；现代子词/字节级 tokenizer 会把罕见词拆成更小单位，降低未知词概率。
# 批处理还需要 `[PAD]` 补齐长度，语言模型通常需要 `[BOS]`、`[EOS]` 表示序列边界，指令模型可能有角色或消息边界 token。
# 保留词 reserve 即表示所有的特殊词
special = {"[PAD]": 0, "[UNK]": 1, "[BOS]": 2, "[EOS]": 3}
normal = {"hello": 4, "world": 5}
vocab = {**special, **normal}
text_tokens = ["[BOS]", "hello", "mystery", "[EOS]"]
ids = [vocab.get(token, vocab["[UNK]"]) for token in text_tokens]
print(ids)

[2, 4, 1, 3]


In [8]:
## 4. Tokenizer：模型看到的文本单位
# Tokenizer 将文本编码成 token ID 序列，也负责把 ID 解码回文本。
# 字符级词表小但序列长；词级可读但词表大、容易遇到新词；子词级在词表规模与覆盖能力之间折中。
# BPE、WordPiece、Unigram 是不同的子词构造/切分方法，真实行为还取决于预切分、规范化和特殊 token 配置。
#
text = "unbelievable"
char_tokens = list(text)
word_tokens = [text]
toy_subword_tokens = ["un", "believ", "able"]  # 仅作示意，非真实 tokenizer 输出
print(char_tokens)
print(word_tokens)
print(toy_subword_tokens)

['u', 'n', 'b', 'e', 'l', 'i', 'e', 'v', 'a', 'b', 'l', 'e']
['unbelievable']
['un', 'believ', 'able']


In [9]:
## 5. 文本分类的传统基线：词袋与 TF-IDF

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

texts = ["good food and good service", "bad food and slow service"]
count = CountVectorizer()
X_count = count.fit_transform(texts)
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(texts)
print(count.get_feature_names_out())
print(X_count.toarray())
print(X_tfidf.toarray().round(2))


['and' 'bad' 'food' 'good' 'service' 'slow']
[[1 0 1 2 1 0]
 [1 1 1 0 1 1]]
[[0.3  0.   0.3  0.85 0.3  0.  ]
 [0.38 0.53 0.38 0.   0.38 0.53]]


In [13]:
## 8. 文本向量化：One-hot 与 Embedding
# One-hot 用长度为词表大小 `V` 的稀疏向量表示 token；
# Embedding 用 `D` 维可训练稠密向量表示 token。Embedding 可以看作查一个形状 `[V,D]` 的参数表。向量的语义结构来自训练目标，而不是整数 ID 本身。
vocab_size, embedding_dim = 10, 5
ids = torch.tensor([[1, 3, 2]]) # 序列长度是3；1,3,2分别表示token_id
embedding = nn.Embedding(vocab_size, embedding_dim)
vectors = embedding(ids)
print(vectors.shape)  # [B=1, L=3, D=5]

torch.Size([1, 3, 5])


In [18]:
# 分类任务常把 token 向量聚合成句向量。平均池化易理解，但应排除 padding；RNN/Transformer 则通过顺序或注意力构造上下文相关表示。Embedding 的训练、梯度、预训练向量和向量相似度详见 Embedding 专题。

pad_id = 0
ids = torch.tensor([[1, 2, 0], [3, 0, 0]])
embedding = nn.Embedding(6, 5, padding_idx=pad_id)
vectors = embedding(ids)                         # [B, L, D]
mask = ids.ne(pad_id).unsqueeze(-1)              # [B, L, 1]
sentence_vectors = (vectors * mask).sum(1) / mask.sum(1).clamp_min(1) # 求平均向量 Mean Pooling（平均池化）
print(sentence_vectors.shape)  # [B, D]



torch.Size([2, 5])
tensor([[ True,  True, False],
        [ True, False, False]])
tensor([[[ True],
         [ True],
         [False]],

        [[ True],
         [False],
         [False]]])


In [ ]:
## 9. 序列长度：padding、mask 与截断
# 一个 batch 中不同文本长度不同，张量要求规则形状，因此通常把较短序列补成相同长度。Padding 是占位符，不是文本内容；若模型或池化把它当作真实 token，表示和 loss 会被污染。
# Attention mask 通常标出有效 token，但不同 API 对 mask 的真假含义可能相反，必须核对接口。

